# M49 Kaggle — fine-tune generator bằng official train

Add đúng hai input: Dataset chứa `legal-agentic-rag-m49-source.zip` và Dataset chính thức chứa `train.json`. Bật Internet và GPU T4 x2. Notebook chỉ fine-tune generator; không dùng public, không build lại DB/index và không tạo dữ liệu tổng hợp.

Training lưu checkpoint mỗi 50 optimizer step. Nếu gần hết session, Quick Save với **Always save output**; lần sau add output notebook này để resume. Khi thấy `M49 OUTPUT` và manifest `complete: true`, hãy Quick Save lần cuối.

In [ ]:
from hashlib import sha256
from pathlib import Path
import runpy
import sys
from zipfile import ZipFile

input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
scripts = sorted(input_root.rglob('m49_kaggle_train_generator.py'))
if not scripts:
    zips = sorted(input_root.rglob('legal-agentic-rag-m49-source.zip'))
    assert len(zips) == 1, f'Cần đúng 1 source ZIP M49, tìm thấy: {zips}'
    bootstrap = working / 'm49-training-bootstrap'
    if not bootstrap.is_dir():
        with ZipFile(zips[0]) as archive:
            archive.extractall(bootstrap)
    scripts = sorted(bootstrap.rglob('m49_kaggle_train_generator.py'))
assert scripts, 'Không tìm thấy training runner M49'
digests = {sha256(path.read_bytes()).hexdigest() for path in scripts}
assert len(digests) == 1, f'Có nhiều source M49 khác nhau: {scripts}'
script = scripts[0]
sys.path.insert(0, str(script.parent))
runpy.run_path(str(script), run_name='__main__')
